# 2. Train and evaluate / 학습과 평가
Train CDLD and MF, adapt target antibiotics with matched supports, and fit LR/RF/XGBoost and a frequency baseline. The inputs, three seeds and support ordering are fixed in config.

CDLD·MF·L2 로지스틱 회귀·RF·XGBoost·빈도 기준을 동일 관측 조건에서 실행합니다. 아래 소스 셀에 학습·적응·외부 추론·채점 구현이 있으며, 마지막 셀이 CPU 작업을 실행합니다. 생성된 모델·예측·진행 기록은 코드 폴더 밖에 저장됩니다.

In [ ]:
from pathlib import Path
import os, json
PACKAGE = Path.cwd().resolve()
assert (PACKAGE/'config/protocol.json').is_file(), 'Open this notebook from the CDLD_AST directory.'
OUTPUT = Path(os.environ.get('CDLD_OUTPUT_DIR', str(Path.home()/'CDLD_AST_outputs'))).expanduser().resolve()
assert OUTPUT != PACKAGE and PACKAGE not in OUTPUT.parents, 'Keep generated files outside the code directory.'
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN = OUTPUT/'run'
RUN.mkdir(exist_ok=True)
import sys, subprocess, shutil
os.environ.update(CUDA_VISIBLE_DEVICES='-1',TF_ENABLE_ONEDNN_OPTS='0',OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',TF_NUM_INTRAOP_THREADS='1',TF_NUM_INTEROP_THREADS='1')
assert sys.platform.startswith('linux'), 'Use the Linux CPU environment specified in requirements.txt.'
assert (RUN/'inputs').is_dir() and (RUN/'labels').is_dir(), 'Run 01_inputs.ipynb first.'
WORKERS = int(os.environ.get('CDLD_WORKERS','1'))
assert WORKERS >= 1
MODULES = {}

## reference/architecture.py

In [ ]:
MODULES['reference/architecture.py'] = r'''
import os
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, concatenate, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Loss
from keras import layers, utils

tf.get_logger().setLevel("ERROR")
for physical_gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(physical_gpu, True)
print("TensorFlow", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))


class ValueOfIndexLayer(tf.keras.layers.Layer):
    def __init__(self, index):
        super(ValueOfIndexLayer, self).__init__()
        self.index = index
        self.value = tf.Variable(initial_value=tf.zeros((1,)))

    def call(self, input):
        self.value = input[:, self.index]
        return self.value

class IndexedValueLayer(layers.Layer):
    def __init__(self, value_shape, trainable: bool, name: str,
                 latent_regularizer=None, latent_constraint=None,
                 latent_initializer="he_normal", **kwargs):
        super(IndexedValueLayer, self).__init__(name=name, **kwargs)
        vocab_size, latent_dims = value_shape
        self.input_dim = vocab_size
        self.output_dim = latent_dims
        self.kernel = self.add_weight(
            shape=(self.input_dim, self.output_dim),
            name=f"{self.name}.kernel", trainable=trainable,
            regularizer=latent_regularizer, constraint=latent_constraint,
            initializer=latent_initializer)

    def call(self, inputs):
        return tf.gather(self.kernel, tf.cast(inputs, tf.int32))

    def set_value(self, value):
        self.kernel.assign(value)

    def get_value(self):
        return self.kernel.numpy()

class BatchBinaryCrossentropy(Loss):
    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
                                                                        
        y_true = tf.reshape(tf.cast(y_true, y_pred.dtype), tf.shape(y_pred))
        bce = -(y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
        return tf.reduce_mean(bce, axis=-1)

def build_finder_model(user_latents_shape, item_features_shape, item_latents_shape,
                       user_latent_trainable=True, item_latent_trainable=True):
    inp = Input((2,), name="input")
    user_index = ValueOfIndexLayer(0)(inp)
    item_index = ValueOfIndexLayer(1)(inp)

    user_latents_layer = IndexedValueLayer(user_latents_shape, trainable=user_latent_trainable, name="user_latents")
    item_features_layer = IndexedValueLayer(item_features_shape, trainable=False, name="item_features")
    item_latents_layer = IndexedValueLayer(item_latents_shape, trainable=item_latent_trainable, name="item_latents")

    user_latent_x = user_latents_layer(user_index)
    item_feature_x = item_features_layer(item_index)
    item_latent_x = item_latents_layer(item_index)

    user_latent_x = Dense(96, activation="swish", use_bias=True, kernel_regularizer=l2(float(os.environ.get("FINDER_L2", "0.0001"))), name="user_latent")(user_latent_x)
    user_latent_x = BatchNormalization()(user_latent_x)
    user_latent_x = Dropout(0.2*float(os.environ.get("FINDER_DROPOUT", "1")))(user_latent_x)

    item_feature_x = Dense(64, activation="swish", use_bias=True, kernel_regularizer=l2(float(os.environ.get("FINDER_L2", "0.0001"))), name="item_feature")(item_feature_x)
    item_feature_x = BatchNormalization()(item_feature_x)
    item_feature_x = Dropout(0.2*float(os.environ.get("FINDER_DROPOUT", "1")))(item_feature_x)

    item_latent_x = Dense(96, activation="swish", use_bias=True, kernel_regularizer=l2(float(os.environ.get("FINDER_L2", "0.0001"))), name="item_latent")(item_latent_x)
    item_latent_x = BatchNormalization()(item_latent_x)
    item_latent_x = Dropout(0.2*float(os.environ.get("FINDER_DROPOUT", "1")))(item_latent_x)

    x = concatenate([user_latent_x, item_feature_x, item_latent_x])
    x = BatchNormalization()(x)
    x = Dense(256, activation="swish", use_bias=True, kernel_regularizer=l2(float(os.environ.get("FINDER_L2", "0.0001"))))(x)
    x = BatchNormalization()(x); x = Dropout(0.3*float(os.environ.get("FINDER_DROPOUT", "1")))(x)
    x = Dense(128, activation="swish", use_bias=True, kernel_regularizer=l2(float(os.environ.get("FINDER_L2", "0.0001"))))(x)
    x = BatchNormalization()(x); x = Dropout(0.3*float(os.environ.get("FINDER_DROPOUT", "1")))(x)
    x = Dense(64, activation="swish", use_bias=True, kernel_regularizer=l2(float(os.environ.get("FINDER_L2", "0.0001"))))(x)
    x = BatchNormalization()(x); x = Dropout(0.2*float(os.environ.get("FINDER_DROPOUT", "1")))(x)
    score_output = Dense(1, activation="sigmoid", kernel_regularizer=l2(float(os.environ.get("FINDER_L2", "0.0001"))))(x)

    model = Model(inp, score_output)
    model.user_latents_layer = user_latents_layer
    model.item_features_layer = item_features_layer
    model.item_latents_layer = item_latents_layer
    return model

def build_predictor_model_optimized(user_feature_dim, item_feature_dim, latent_dim, name="Predictor"):
    user_latent_input = Input((latent_dim,), name="user_latent_input")
    item_feature_input = Input((item_feature_dim,), name="item_feature_input")
    item_latent_input = Input((latent_dim,), name="item_latent_input")
    user_feature_input = Input((user_feature_dim,), name="user_feature_input")

    user_latent_x = Dense(96, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="user_latent")(user_latent_input)
    user_latent_x = BatchNormalization()(user_latent_x); user_latent_x = Dropout(0.2)(user_latent_x)

    item_feature_x = Dense(64, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="item_feature")(item_feature_input)
    item_feature_x = BatchNormalization()(item_feature_x); item_feature_x = Dropout(0.2)(item_feature_x)

    item_latent_x = Dense(96, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="item_latent")(item_latent_input)
    item_latent_x = BatchNormalization()(item_latent_x); item_latent_x = Dropout(0.2)(item_latent_x)

    user_feature_x = Dense(4, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001), name="user_feature")(user_feature_input)
    user_feature_x = BatchNormalization()(user_feature_x); user_feature_x = Dropout(0.1)(user_feature_x)

    x = concatenate([user_latent_x, item_feature_x, item_latent_x, user_feature_x])
    x = BatchNormalization()(x)
    x = Dense(256, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x); x = Dropout(0.3)(x)
    x = Dense(128, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x); x = Dropout(0.3)(x)
    x = Dense(64, activation="swish", use_bias=True, kernel_regularizer=l2(0.0001))(x)
    x = BatchNormalization()(x); x = Dropout(0.2)(x)
    score_output = Dense(1, activation="sigmoid", kernel_regularizer=l2(0.0001))(x)

    return Model(inputs=[user_latent_input, item_feature_input, item_latent_input, user_feature_input],
                 outputs=score_output, name=name)
'''

## reference/predictor.py

In [ ]:
MODULES['reference/predictor.py'] = r'''


class BCEPredictor(tf.keras.Model):
    """Use the reference Predictor objective: BCE, without declared L2 terms."""
    def __init__(self, inputs, outputs):
        super().__init__(inputs=inputs, outputs=outputs)
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.accuracy_tracker = tf.keras.metrics.BinaryAccuracy(name="accuracy")
        self.auc_tracker = tf.keras.metrics.AUC(name="auc")
    @property
    def metrics(self):
        return [self.loss_tracker, self.accuracy_tracker, self.auc_tracker]
    def train_step(self, batch):
        inputs, labels, weights = tf.keras.utils.unpack_x_y_sample_weight(batch)
        with tf.GradientTape() as tape:
            probabilities = self(inputs, training=True)
            loss_value = self.loss(labels, probabilities)
            if getattr(self, "predictor_l2", 0.0):
                loss_value = loss_value + (self.predictor_l2 / 0.0001) * tf.add_n(self.losses)
        gradients = tape.gradient(loss_value, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))
        return self.update_metrics(labels, probabilities, loss_value)
    def test_step(self, batch):
        inputs, labels, weights = tf.keras.utils.unpack_x_y_sample_weight(batch)
        probabilities = self(inputs, training=False)
        return self.update_metrics(labels, probabilities, self.loss(labels, probabilities))
    def update_metrics(self, labels, probabilities, loss_value):
        self.loss_tracker.update_state(loss_value)
        self.accuracy_tracker.update_state(labels, probabilities)
        self.auc_tracker.update_state(labels, probabilities)
        return {metric.name:metric.result() for metric in self.metrics}


def make_indexed_predictor(user_latents, item_latents, config):
    inputs = Input((2,), name="response_indices")
    user_index, item_index = ValueOfIndexLayer(0)(inputs), ValueOfIndexLayer(1)(inputs)
    user_layer = IndexedValueLayer(user_latents.shape, trainable=False, name="fixed_user_latents")
    item_layer = IndexedValueLayer(item_latents.shape, trainable=False, name="fixed_item_latents")
    zeros = layers.Lambda(lambda values: tf.zeros_like(values[:, :1]), name="zero_features")(inputs)
    core = build_predictor_model_optimized(1, 1, config["latent_size"])
    output = core([user_layer(user_index), zeros, item_layer(item_index), zeros])
    model = BCEPredictor(inputs, output)
    user_layer.set_value(user_latents)
    item_layer.set_value(item_latents)
    model.user_latents_layer, model.item_latents_layer = user_layer, item_layer
    return model


                                                                              
CHECKPOINT_DIRECTORY = None

def save_training_state(model, label, include_optimizer=True):
    if CHECKPOINT_DIRECTORY is None:
        return
    values = {f"model_{j}":v.numpy() for j,v in enumerate(model.variables)}
    metadata = {"model":[{"name":v.path,"shape":list(v.shape)} for v in model.variables]}
    if include_optimizer:
        values.update({f"optimizer_{j}":v.numpy() for j,v in enumerate(model.optimizer.variables)})
        metadata["optimizer"] = [{"name":v.path,"shape":list(v.shape)} for v in model.optimizer.variables]
    else:
        metadata["optimizer_note"] = "Predictor best weights restored by original callback; end optimizer is not paired with them and is not exported."
    path = CHECKPOINT_DIRECTORY / (label + "_state.npz")
    temporary = path.with_suffix(".pending.npz")
    np.savez(temporary, **values)
    temporary.replace(path)
    save_json(CHECKPOINT_DIRECTORY / (label + "_state.json"), metadata)

OriginalTrainingProgress = TrainingProgress
'''

## code/train_cdld.py

In [ ]:
MODULES['code/train_cdld.py'] = r'''
"""Train Finder and Predictor at the fixed endpoints."""
import os
os.environ['TF_NUM_INTRAOP_THREADS']='1';os.environ['TF_NUM_INTEROP_THREADS']='1';os.environ['TF_CPP_MIN_LOG_LEVEL']='3'
from pathlib import Path
import sys,json,time,hashlib,traceback
import numpy as np
import tensorflow as tf
from checkpoint import save_checkpoint,restore_checkpoint
H=Path(__file__).resolve().parents[1];REF=H/'reference'
exec(compile((REF/'architecture.py').read_text(),str(REF/'architecture.py'),'exec'))
exec(compile((REF/'predictor.py').read_text(),str(REF/'predictor.py'),'exec'))
def write(p,x):
 t=p.with_suffix('.tmp');t.write_text(json.dumps(x,indent=2));t.replace(p)
def bce(y,p):
 p=np.clip(np.asarray(p).ravel(),1e-7,1-1e-7);y=np.asarray(y).ravel()
 return float(-(y*np.log(p)+(1-y)*np.log1p(-p)).mean())
def visit(m):
 for layer in m.layers:
  yield layer
  if hasattr(layer,'layers'):yield from visit(layer)
def train(split,out,cfg):
 out.mkdir(parents=True,exist_ok=True);start=time.monotonic();write(out/'config.json',cfg)
 z=np.load(split);x,y,vx,vy=[z[k] for k in ['train_x','train_y','val_x','val_y']]
 nu=int(max(x[:,0].max(),vx[:,0].max()))+1;ni=int(max(x[:,1].max(),vx[:,1].max()))+1
 seed=cfg['seed'];dim=cfg['latent_size'];tf.keras.utils.set_random_seed(seed);rng=np.random.default_rng(seed)
 iu=rng.random((nu,dim)).astype('float32');ii=rng.random((ni,dim)).astype('float32')
 uld=build_finder_model(iu.shape,(ni,1),ii.shape,True,False);ild=build_finder_model(iu.shape,(ni,1),ii.shape,False,True)
 for m in [uld,ild]:
  m.user_latents_layer.set_value(iu);m.item_latents_layer.set_value(ii);m.item_features_layer.set_value(np.zeros((ni,1),'float32'));m.compile(optimizer=tf.keras.optimizers.Adam(cfg['finder_lr']),loss=BatchBinaryCrossentropy())
 models={'ULD':uld,'ILD':ild};hist=[]
 def pred(m,a):return np.concatenate([m(a[i:i+cfg['batch_size']],training=False).numpy().ravel() for i in range(0,len(a),cfg['batch_size'])])
 def epoch(m):
  order=rng.permutation(len(x))
  for i in range(0,len(x),cfg['batch_size']):
   b=order[i:i+cfg['batch_size']];m.train_on_batch(x[b],y[b])
 for cycle in range(1,max(cfg['finder_cycles'])+1):
  write(out/'status.json',dict(phase='finder',cycle=cycle));uld.item_latents_layer.set_value(ild.item_latents_layer.get_value())
  for name,m in models.items():
   if name=='ILD':ild.user_latents_layer.set_value(uld.user_latents_layer.get_value())
   for _ in range(cfg['finder_subepochs']):epoch(m)
  uld.item_latents_layer.set_value(ild.item_latents_layer.get_value());ub=bce(vy,pred(uld,vx));ib=bce(vy,pred(ild,vx));ut=bce(y,pred(uld,x));it=bce(y,pred(ild,x))
  row=dict(cycle=cycle,epochs_per_component=cycle*cfg['finder_subepochs'],ULD_validation_bce=ub,ILD_validation_bce=ib,mean_validation_bce=(ub+ib)/2,ULD_train_bce=ut,ILD_train_bce=it,mean_train_bce=(ut+it)/2);hist.append(row)
  if cycle in cfg['finder_cycles']:save_checkpoint(out/f'finder_{cycle:02d}',models,cfg,dict(cycle=cycle),rng)
  write(out/'finder_history.json',hist);print('Finder',cycle,row['mean_validation_bce'],flush=True)
 for cycle in cfg['finder_cycles']:
  restore_checkpoint(out/f'finder_{cycle:02d}',models,cfg,rng)
  assert abs(bce(vy,pred(ild,vx))-hist[cycle-1]['ILD_validation_bce'])<1e-7
  if not cfg['predictor_epochs']:continue
  ul=uld.user_latents_layer.get_value();il=ild.item_latents_layer.get_value();tf.keras.utils.set_random_seed(seed);rng=np.random.default_rng(seed)
  m=make_indexed_predictor(ul,il,cfg)
  m.predictor_l2=cfg.get("predictor_l2",0.0)
  for l in visit(m):
   if isinstance(l,tf.keras.layers.BatchNormalization):l.momentum=cfg['predictor_bn_momentum']
  m.compile(optimizer=tf.keras.optimizers.Adam(cfg['predictor_lr']),loss=BatchBinaryCrossentropy());ph=[]
  for e in range(1,max(cfg['predictor_epochs'])+1):
   epoch(m);np.testing.assert_array_equal(m.user_latents_layer.get_value(),ul);np.testing.assert_array_equal(m.item_latents_layer.get_value(),il)
   ph.append(dict(epoch=e,train_bce=bce(y,m(x,training=False).numpy()),validation_bce=bce(vy,m(vx,training=False).numpy())))
   if e in cfg['predictor_epochs']:
    save_checkpoint(out/f'predictor_{cycle:02d}_{e:03d}',{'predictor':m},cfg,dict(cycle=cycle,epoch=e),rng)
    np.savez_compressed(out/f'validation_{cycle:02d}_{e:03d}.npz',probability=m(vx,training=False).numpy().ravel())
   write(out/f'predictor_history_{cycle:02d}.json',ph)
   write(out/'status.json',dict(phase='predictor',cycle=cycle,epoch=e))
  write(out/f'predictor_history_{cycle:02d}.json',ph);print('Predictor complete for',cycle,flush=True)
 write(out/'completion.json',dict(passed=True,seconds=time.monotonic()-start,split_sha256=hashlib.sha256(Path(split).read_bytes()).hexdigest(),source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()))
 write(out/'status.json',dict(phase='training_complete'))
if __name__=='__main__':
 split,out,config=map(Path,sys.argv[1:4])
 try:train(split,out,json.loads(config.read_text()))
 except Exception:
  out.mkdir(parents=True,exist_ok=True);write(out/'failure.json',dict(traceback=traceback.format_exc()));raise
'''

## code/checkpoint.py

In [ ]:
MODULES['code/checkpoint.py'] = r'''
"""Atomic model/optimizer/RNG checkpoints with config identity checking."""
from pathlib import Path
import hashlib,json,os,random
import numpy as np

def digest(config):
    return hashlib.sha256(json.dumps(config,sort_keys=True,separators=(',',':')).encode()).hexdigest()

def save_checkpoint(path,models,config,progress,rng):
    path=Path(path);path.mkdir(parents=True,exist_ok=True)
    arrays={};layout={}
    for name,model in models.items():
        layout[name]={}
        for kind,variables in [('model',model.variables),('optimizer',model.optimizer.variables)]:
            layout[name][kind]=[list(v.shape) for v in variables]
            for i,value in enumerate(variables):arrays[f'{name}.{kind}.{i}']=value.numpy()
    temporary=path/'state.tmp.npz';np.savez_compressed(temporary,**arrays)
    sha=hashlib.sha256(temporary.read_bytes()).hexdigest()
                                                                                                        
    payload=f'state-{sha}.npz';os.replace(temporary,path/payload)
    legacy=np.random.get_state()
    metadata={'payload':payload,'sha256':sha,'layout':layout,'config_hash':digest(config),'config':config,'progress':progress,
              'rng':rng.bit_generator.state,'python_rng':random.getstate(),
              'numpy_rng':[legacy[0],legacy[1].tolist(),legacy[2],legacy[3],legacy[4]]}
    temporary=path/'checkpoint.tmp.json';temporary.write_text(json.dumps(metadata,indent=2));os.replace(temporary,path/'checkpoint.json')

def restore_checkpoint(path,models,config,rng):
    path=Path(path);meta=json.loads((path/'checkpoint.json').read_text())
    if meta['config_hash']!=digest(config):raise ValueError('Checkpoint configuration differs.')
    payload=path/meta['payload']
    if hashlib.sha256(payload.read_bytes()).hexdigest()!=meta['sha256']:raise ValueError('Checkpoint checksum mismatch.')
    with np.load(payload,allow_pickle=False) as arrays:
        pending=[]
        for name,model in models.items():
            model.optimizer.build(model.trainable_variables)
            for kind,variables in [('model',model.variables),('optimizer',model.optimizer.variables)]:
                if [list(v.shape) for v in variables]!=meta['layout'][name][kind]:raise ValueError('Variable layout differs.')
                pending.extend((v,arrays[f'{name}.{kind}.{i}']) for i,v in enumerate(variables))
        if set(models)!=set(meta['layout']):raise ValueError('Model set differs.')
        for variable,value in pending:variable.assign(value)
    def tuples(value):return tuple(tuples(x) for x in value) if isinstance(value,list) else value
    random.setstate(tuples(meta['python_rng']))
    legacy=meta['numpy_rng'];np.random.set_state((legacy[0],np.array(legacy[1],np.uint32),legacy[2],legacy[3],legacy[4]))
    rng.bit_generator.state=meta['rng']
    return meta['progress']
'''

## code/expand_model.py

In [ ]:
MODULES['code/expand_model.py'] = r'''
"""Expand only row tables, preserving all original network variables."""
import numpy as np

def transplant_variables(source,destination,expanded_values):
    if len(source.variables)!=len(destination.variables):raise ValueError('Variable count mismatch.')
    for old,new in zip(source.variables,destination.variables):
        value=expanded_values.get(id(new),old.numpy())
        if tuple(new.shape)!=value.shape:raise ValueError(f'Unexpected variable shape: {new.shape} vs {value.shape}')
        new.assign(value)

def assert_old_predictions(source,destination,inputs):
    a=source(inputs,training=False).numpy();b=destination(inputs,training=False).numpy()
    np.testing.assert_allclose(a,b,atol=1e-7,rtol=0)
    return float(np.max(np.abs(a-b)))
'''

## code/adaptation.py

In [ ]:
MODULES['code/adaptation.py'] = r'''
"""Frozen model adaptation utilities."""
import os
os.environ['TF_NUM_INTRAOP_THREADS']='1';os.environ['TF_NUM_INTEROP_THREADS']='1';os.environ['TF_CPP_MIN_LOG_LEVEL']='3'
from pathlib import Path
import sys,json,time,hashlib,argparse,traceback
import numpy as np
import tensorflow as tf
import train_cdld as ref
from expand_model import transplant_variables,assert_old_predictions
from checkpoint import restore_checkpoint,save_checkpoint
H=Path(__file__).resolve().parents[1]
def write(p,v):
 t=p.with_suffix('.tmp');t.write_text(json.dumps(v,indent=2));t.replace(p)
def adapting_predictor(ul,il,cfg):
 inp=ref.Input((2,),name='response_indices');u=ref.ValueOfIndexLayer(0)(inp);i=ref.ValueOfIndexLayer(1)(inp)
 a=ref.IndexedValueLayer(ul.shape,False,'fixed_user_latents');b=ref.IndexedValueLayer(il.shape,True,'fixed_item_latents')
 zeros=ref.layers.Lambda(lambda x:tf.zeros_like(x[:,:1]),name='zero_features')(inp)
 core=ref.build_predictor_model_optimized(1,1,cfg['latent_size']);out=core([a(u),zeros,b(i),zeros]);m=ref.BCEPredictor(inp,out)
 m.user_latents_layer=a;m.item_latents_layer=b;a.set_value(ul);b.set_value(il);return m
def freeze(m,layer,lr):
 for l in m.layers:l.trainable=False
 layer.trainable=True;m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss=ref.BatchBinaryCrossentropy());m.optimizer.build(m.trainable_weights)
 assert len(m.trainable_weights)==1 and m.trainable_weights[0] is layer.kernel
 return [v.numpy().copy() for v in m.variables],[v.numpy().copy() for v in m.optimizer.variables]
def reset(m,variables,optimizer,lr):
 for a,b in zip(m.variables,variables):a.assign(b)
 for a,b in zip(m.optimizer.variables,optimizer):a.assign(b)
 m.optimizer.learning_rate.assign(lr)
def check_frozen(m,initial,active,old_items=None):
 for v,a in zip(m.variables,initial):
                                                                      
  if v is active.kernel or 'seed_generator' in v.path:continue
  np.testing.assert_array_equal(v.numpy(),a)
 if old_items is not None:np.testing.assert_array_equal(active.get_value()[:-1],old_items)
def make_chunk(m,initial,ni):
 fixed=[(v,tf.constant(a)) for v,a in zip(m.variables,initial) if v is not m.item_latents_layer.kernel and 'seed_generator' not in v.path]
 old=tf.constant(m.item_latents_layer.get_value()[:ni])
 @tf.function(input_signature=[tf.TensorSpec([None,2],tf.float32),tf.TensorSpec([None,1],tf.float32),tf.TensorSpec([],tf.int32)])
 def chunk(x,y,n):
  losses=tf.TensorArray(tf.float32,size=n)
  for i in tf.range(n):
   for metric in m.metrics:metric.reset_state()
   result=m.train_step((x,y));losses=losses.write(i,tf.cast(result['loss'],tf.float32))
   for var,value in fixed:tf.debugging.assert_equal(var,value)
   tf.debugging.assert_equal(m.item_latents_layer.kernel[:ni],old)
  return losses.stack()
 return chunk
'''

## code/adapt_cdld.py

In [ ]:
MODULES['code/adapt_cdld.py'] = r'''
"""Adapt one target-drug latent using support labels."""
from pathlib import Path
import json,sys,time,hashlib,traceback
from adaptation import ref,tf,np,freeze,reset,check_frozen,make_chunk,transplant_variables,assert_old_predictions,restore_checkpoint,adapting_predictor,write
H=Path(__file__).resolve().parents[1]
def run(folder,seed,cycle,epoch,inputpack,out,casespath):
 out.mkdir(parents=True,exist_ok=True);start=time.monotonic();trained=folder/f'seed{seed}';cfg=json.loads((trained/'config.json').read_text());entities=json.loads((folder/'entities.json').read_text());nu=len(entities['episodes']);ni=len(entities['drugs']);dim=cfg['latent_size']
 inp=np.load(inputpack);oldcheck=np.load(folder/'split.npz')['val_x'][:128];rng=np.random.default_rng(seed);tf.keras.utils.set_random_seed(seed)
 uld=ref.build_finder_model((nu,dim),(ni,1),(ni,dim),True,False);ild=ref.build_finder_model((nu,dim),(ni,1),(ni,dim),False,True)
 for m in [uld,ild]:m.compile(optimizer=tf.keras.optimizers.Adam(cfg['finder_lr']),loss=ref.BatchBinaryCrossentropy())
 restore_checkpoint(trained/f'finder_{cycle:02d}',{'ULD':uld,'ILD':ild},cfg,rng)
 ul=np.vstack([uld.user_latents_layer.get_value(),np.random.default_rng(seed).random((int(inp['new_rows']),dim)).astype('float32')]);il=np.vstack([ild.item_latents_layer.get_value(),np.random.default_rng(seed+1000).random((1,dim)).astype('float32')])
 pred=ref.make_indexed_predictor(ul[:nu],il[:ni],cfg)
 pred.compile(optimizer=tf.keras.optimizers.Adam(cfg['predictor_lr']),loss=ref.BatchBinaryCrossentropy());restore_checkpoint(trained/f'predictor_{cycle:02d}_{epoch:03d}',{'predictor':pred},cfg,rng)
 expanded=[]
 for old,flags in [(uld,(True,False)),(ild,(False,True))]:
  m=ref.build_finder_model(ul.shape,(ni+1,1),il.shape,*flags)
  transplant_variables(old,m,{id(m.user_latents_layer.kernel):ul,id(m.item_latents_layer.kernel):il,id(m.item_features_layer.kernel):np.zeros((ni+1,1),'float32')});assert_old_predictions(old,m,oldcheck);expanded.append(m)
 uld,ild=expanded
 p=adapting_predictor(ul,il,cfg);transplant_variables(pred,p,{id(p.user_latents_layer.kernel):ul,id(p.item_latents_layer.kernel):il});assert_old_predictions(pred,p,oldcheck)
 initial,_=freeze(uld,uld.user_latents_layer,cfg['row_lr'])
 for step in range(cfg['row_steps']):
  uld.train_on_batch(inp['observed_x'],inp['observed_y'].reshape(-1,1));check_frozen(uld,initial,uld.user_latents_layer)
  np.testing.assert_array_equal(uld.user_latents_layer.get_value()[:nu],ul[:nu])
 users=uld.user_latents_layer.get_value();ild.user_latents_layer.set_value(users);p.user_latents_layer.set_value(users)
 np.savez_compressed(out/'fixed_rows.npz',users=users)

 cases=json.loads(casespath.read_text());assert cases and len({x['id'] for x in cases})==len(cases)
 snapshots={arm:freeze(m,m.item_latents_layer,.0005) for arm,m in [('Predictor_BCE',p)]}
 chunks={arm:make_chunk(m,snapshots[arm][0],ni) for arm,m in [('Predictor_BCE',p)]}
 for rep in range(10):
  for case in cases:
   arm=case['arm'];budget=case['budget'];lr=case['lr'];step=case['step'];assert arm in snapshots and budget in cfg['support_budgets'] and lr in cfg['adapt_lrs'] and step in cfg['adapt_steps']
   m=p;initial,opt=snapshots[arm];reset(m,initial,opt,lr);sx=inp[f'x_{rep}_{budget}'] if budget else np.empty((0,2),'float32');sy=inp[f'y_{rep}_{budget}'].reshape(-1,1) if budget else np.empty((0,1),'float32')
   if step:chunks[arm](sx,sy,np.int32(step)).numpy()
   check_frozen(m,initial,m.item_latents_layer,il[:ni]);items=m.item_latents_layer.get_value().copy()
   prob=p(inp['query_x'],training=False).numpy().ravel();assert np.isfinite(prob).all()
   dest=out/f"r{rep}_{case['id']}.npz";tmp=dest.with_suffix('.tmp.npz');np.savez_compressed(tmp,probability=prob,new_drug_latent=items[-1]);tmp.replace(dest)
 write(out/'completion.json',dict(passed=True,cases=len(cases),repeats=10,seconds=time.monotonic()-start,input_sha256=hashlib.sha256(inputpack.read_bytes()).hexdigest(),cases_sha256=hashlib.sha256(casespath.read_bytes()).hexdigest(),source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),query_labels_read=False))
if __name__=='__main__':
 try:run(Path(sys.argv[1]),int(sys.argv[2]),int(sys.argv[3]),int(sys.argv[4]),Path(sys.argv[5]),Path(sys.argv[6]),Path(sys.argv[7]))
 except Exception:
  out=Path(sys.argv[6]);out.mkdir(parents=True,exist_ok=True);write(out/'failure.json',dict(traceback=traceback.format_exc()));raise
'''

## code/external_cdld.py

In [ ]:
MODULES['code/external_cdld.py'] = r'''
"""Predict external cases using the MIMIC-adapted target latent."""
from pathlib import Path
import json,sys,time,hashlib,traceback
from adaptation import ref,tf,np,freeze,reset,check_frozen,make_chunk,transplant_variables,assert_old_predictions,restore_checkpoint,adapting_predictor,write
H=Path(__file__).resolve().parents[1]
def run(folder,seed,cycle,epoch,inputpack,out,casespath,adapted):
 out.mkdir(parents=True,exist_ok=True);start=time.monotonic();trained=folder/f'seed{seed}';cfg=json.loads((trained/'config.json').read_text());entities=json.loads((folder/'entities.json').read_text());nu=len(entities['episodes']);ni=len(entities['drugs']);dim=cfg['latent_size']
 inp=np.load(inputpack);oldcheck=np.load(folder/'split.npz')['val_x'][:128];rng=np.random.default_rng(seed);tf.keras.utils.set_random_seed(seed)
 uld=ref.build_finder_model((nu,dim),(ni,1),(ni,dim),True,False);ild=ref.build_finder_model((nu,dim),(ni,1),(ni,dim),False,True)
 for m in [uld,ild]:m.compile(optimizer=tf.keras.optimizers.Adam(cfg['finder_lr']),loss=ref.BatchBinaryCrossentropy())
 restore_checkpoint(trained/f'finder_{cycle:02d}',{'ULD':uld,'ILD':ild},cfg,rng)
 ul=np.vstack([uld.user_latents_layer.get_value(),np.random.default_rng(seed).random((int(inp['new_rows']),dim)).astype('float32')]);il=np.vstack([ild.item_latents_layer.get_value(),np.random.default_rng(seed+1000).random((1,dim)).astype('float32')])
 pred=ref.make_indexed_predictor(ul[:nu],il[:ni],cfg)
 pred.compile(optimizer=tf.keras.optimizers.Adam(cfg['predictor_lr']),loss=ref.BatchBinaryCrossentropy());restore_checkpoint(trained/f'predictor_{cycle:02d}_{epoch:03d}',{'predictor':pred},cfg,rng)
 expanded=[]
 for old,flags in [(uld,(True,False)),(ild,(False,True))]:
  m=ref.build_finder_model(ul.shape,(ni+1,1),il.shape,*flags)
  transplant_variables(old,m,{id(m.user_latents_layer.kernel):ul,id(m.item_latents_layer.kernel):il,id(m.item_features_layer.kernel):np.zeros((ni+1,1),'float32')});assert_old_predictions(old,m,oldcheck);expanded.append(m)
 uld,ild=expanded
 p=adapting_predictor(ul,il,cfg);transplant_variables(pred,p,{id(p.user_latents_layer.kernel):ul,id(p.item_latents_layer.kernel):il});assert_old_predictions(pred,p,oldcheck)
 initial,_=freeze(uld,uld.user_latents_layer,cfg['row_lr'])
 for step in range(cfg['row_steps']):
  uld.train_on_batch(inp['observed_x'],inp['observed_y'].reshape(-1,1));check_frozen(uld,initial,uld.user_latents_layer)
  np.testing.assert_array_equal(uld.user_latents_layer.get_value()[:nu],ul[:nu])
 users=uld.user_latents_layer.get_value();ild.user_latents_layer.set_value(users);p.user_latents_layer.set_value(users)
 np.savez_compressed(out/'fixed_rows.npz',users=users)

 cases=json.loads(casespath.read_text());base=p.item_latents_layer.get_value().copy()
 for rep in range(10):
  for case in cases:
   values=np.load(adapted/f"r{rep}_{case['id']}.npz");items=base.copy();items[-1]=values['new_drug_latent'];p.item_latents_layer.set_value(items)
   prob=p(inp['query_x'],training=False).numpy().ravel();assert np.isfinite(prob).all();np.testing.assert_array_equal(p.item_latents_layer.get_value(),items)
   np.savez_compressed(out/f"r{rep}_{case['id']}.npz",probability=prob)
 write(out/'completion.json',dict(passed=True,cases=len(cases),repeats=10,query_labels_read=False,target_drug_updates=0,seconds=time.monotonic()-start,input_sha256=hashlib.sha256(inputpack.read_bytes()).hexdigest(),source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()))
if __name__=='__main__':
 try:run(Path(sys.argv[1]),int(sys.argv[2]),int(sys.argv[3]),int(sys.argv[4]),Path(sys.argv[5]),Path(sys.argv[6]),Path(sys.argv[7]),Path(sys.argv[8]))
 except Exception:
  out=Path(sys.argv[6]);out.mkdir(parents=True,exist_ok=True);write(out/'failure.json',dict(traceback=traceback.format_exc()));raise
'''

## mf_code/train_mf.py

In [ ]:
MODULES['mf_code/train_mf.py'] = r'''
"""Train the selected matrix factorization model."""
import os
os.environ['TF_NUM_INTRAOP_THREADS']='1'
os.environ['TF_NUM_INTEROP_THREADS']='1'
os.environ['TF_CPP_MIN_LOG_LEVEL']='3'
from pathlib import Path
import json,sys,time,hashlib,traceback
import numpy as np
import tensorflow as tf
from sklearn.metrics import log_loss
H=Path(__file__).resolve().parents[1]
def main():
 splitpath,out,seed=Path(sys.argv[1]),Path(sys.argv[2]),int(sys.argv[3])
 out.mkdir(parents=True,exist_ok=True);start=time.time()
 cfg=json.loads((H/'mf_training.json').read_text())['MF']
 checkpoints=cfg['checkpoints'] if len(sys.argv)<5 else [int(sys.argv[4])]
 rng=np.random.default_rng(seed);tf.keras.utils.set_random_seed(seed)
 z=np.load(splitpath);tx=z['train_x'].astype(int);ty=z['train_y'].ravel();vx=z['val_x'].astype(int);vy=z['val_y'].ravel()
 nu=int(max(tx[:,0].max(),vx[:,0].max()))+1;ni=int(max(tx[:,1].max(),vx[:,1].max()))+1
 u=tf.Variable(rng.normal(0,.1,(nu,64)).astype('float32'));v=tf.Variable(rng.normal(0,.1,(ni,64)).astype('float32'))
 ub=tf.Variable(np.zeros(nu,'float32'));vb=tf.Variable(np.zeros(ni,'float32'));bias=tf.Variable(np.float32(np.log(ty.mean()/(1-ty.mean()))))
 variables=[u,v,ub,vb,bias];opt=tf.keras.optimizers.Adam(.01)
 def logits(x):return tf.reduce_sum(tf.gather(u,x[:,0])*tf.gather(v,x[:,1]),axis=1)+tf.gather(ub,x[:,0])+tf.gather(vb,x[:,1])+bias
 @tf.function
 def step():
  with tf.GradientTape() as tape:
   loss=tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=ty,logits=logits(tx)))+.001*(tf.reduce_mean(u*u)+tf.reduce_mean(v*v))
  opt.apply_gradients(zip(tape.gradient(loss,variables),variables))
 history=[]
 for i in range(1,max(checkpoints)+1):
  step()
  if i in checkpoints:
   prob=tf.sigmoid(logits(vx)).numpy()
   np.savez_compressed(out/f'checkpoint_{i}.npz',**dict(zip(['u','v','ub','vb','bias'],[a.numpy() for a in variables])),**{f'optimizer_{j}':a.numpy() for j,a in enumerate(opt.variables)})
   (out/f'checkpoint_{i}.json').write_text(json.dumps(dict(rng=rng.bit_generator.state,step=i)))
   np.savez_compressed(out/f'validation_{i}.npz',probability=prob)
   history.append(dict(step=i,validation_bce=float(log_loss(vy,prob,labels=[0,1]))))
   (out/'history.json').write_text(json.dumps(history,indent=2))
   (out/'status.json').write_text(json.dumps(dict(phase='training',step=i)))
 (out/'completion.json').write_text(json.dumps(dict(passed=True,seconds=time.time()-start,seed=seed,checkpoints=checkpoints,split_sha256=hashlib.sha256(splitpath.read_bytes()).hexdigest(),script_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()),indent=2))
if __name__=='__main__':
 try:main()
 except Exception:
  out=Path(sys.argv[2]);out.mkdir(parents=True,exist_ok=True);(out/'failure.json').write_text(json.dumps(dict(traceback=traceback.format_exc())));raise
'''

## mf_code/adapt_mf.py

In [ ]:
MODULES['mf_code/adapt_mf.py'] = r'''
"""Adapt MF case and target-drug factors."""
import os
os.environ['TF_NUM_INTRAOP_THREADS']='1';os.environ['TF_NUM_INTEROP_THREADS']='1';os.environ['TF_CPP_MIN_LOG_LEVEL']='3'
from pathlib import Path
import json,sys,time,hashlib,traceback
import numpy as np
import tensorflow as tf
H=Path(__file__).resolve().parents[1]
def main():
 checkpoint,inputpack,out,casespath=map(Path,sys.argv[1:5]);out.mkdir(parents=True,exist_ok=True);start=time.time()
 cfg=json.loads((H/'mf_adaptation.json').read_text())
 z=np.load(inputpack);a=np.load(checkpoint.with_suffix('.npz'));rng=np.random.default_rng();rng.bit_generator.state=json.loads(checkpoint.with_suffix('.json').read_text())['rng']
 u,v,ub,vb,bias=[tf.constant(a[k]) for k in ['u','v','ub','vb','bias']];nu=u.shape[0]
 ox=z['observed_x'].astype(int);oy=z['observed_y'].ravel().astype('float32');qx=z['query_x'].astype(int)
 n=int(max(ox[:,0].max(),qx[:,0].max()))+1-nu
 new_u=tf.Variable(rng.normal(0,.1,(n,64)).astype('float32'));new_b=tf.Variable(np.zeros(n,'float32'));ro=tf.keras.optimizers.Adam(.01)
 @tf.function
 def row_step():
  with tf.GradientTape() as tape:
   logits=tf.reduce_sum(tf.gather(new_u,ox[:,0]-nu)*tf.gather(v,ox[:,1]),axis=1)+tf.gather(new_b,ox[:,0]-nu)+tf.gather(vb,ox[:,1])+bias
   loss=tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=oy,logits=logits))+.001*tf.reduce_mean(new_u*new_u)
  ro.apply_gradients(zip(tape.gradient(loss,[new_u,new_b]),[new_u,new_b]))
 for _ in range(300):row_step()
 fixed_u=new_u.numpy().copy();fixed_b=new_b.numpy().copy();np.savez_compressed(out/'fixed_rows.npz',u=fixed_u,ub=fixed_b)
 fu=tf.constant(fixed_u);fb=tf.constant(fixed_b)
 nv=tf.Variable(rng.normal(0,.1,(64,)).astype('float32'));nb=tf.Variable(np.float32(0));initv=nv.numpy().copy();initb=nb.numpy().copy()
 opt=tf.keras.optimizers.Adam(.01);opt.build([nv,nb]);optinit=[x.numpy().copy() for x in opt.variables]
 @tf.function
 def col_step(ids,y):
  with tf.GradientTape() as tape:
   logits=tf.linalg.matvec(tf.gather(fu,ids),nv)+tf.gather(fb,ids)+nb+bias
   loss=tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y,logits=logits))+.001*tf.reduce_mean(nv*nv)
  opt.apply_gradients(zip(tape.gradient(loss,[nv,nb]),[nv,nb]))
  return loss
 def predict():return tf.sigmoid(tf.linalg.matvec(tf.gather(new_u,qx[:,0]-nu),nv)+tf.gather(new_b,qx[:,0]-nu)+nb+bias).numpy()
 cases=json.loads(casespath.read_text());count=0
 for r in range(10):
  for case in cases:
   b,lr,step=case['budget'],case['lr'],case['step'];assert b in [0,10,20,30,40,50] and lr in cfg['MF']['drug_lrs'] and step in cfg['selection_steps']
   x=z[f'x_{r}_{b}'].astype(int) if b else np.empty((0,2),int);y=z[f'y_{r}_{b}'].ravel().astype('float32') if b else np.empty(0,'float32');ids=(x[:,0]-nu).astype('int64');nv.assign(initv);nb.assign(initb)
   for variable,value in zip(opt.variables,optinit):variable.assign(value)
   opt.learning_rate.assign(lr)
   for _ in range(step):col_step(ids,y)
   prob=predict();assert np.isfinite(prob).all();np.testing.assert_array_equal(new_u.numpy(),fixed_u);np.testing.assert_array_equal(new_b.numpy(),fixed_b)
   dest=out/f"r{r}_{case['id']}.npz";tmp=dest.with_suffix('.tmp.npz');np.savez_compressed(tmp,probability=prob,new_drug_latent=nv.numpy(),new_drug_bias=nb.numpy());tmp.replace(dest);count+=1
 (out/'completion.json').write_text(json.dumps(dict(passed=True,conditions=count,seconds=time.time()-start,query_labels_read=False,input_sha256=hashlib.sha256(inputpack.read_bytes()).hexdigest(),cases_sha256=hashlib.sha256(casespath.read_bytes()).hexdigest(),source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()),indent=2))
if __name__=='__main__':
 try:main()
 except Exception:
  out=Path(sys.argv[3]);out.mkdir(parents=True,exist_ok=True);(out/'failure.json').write_text(json.dumps(dict(traceback=traceback.format_exc())));raise
'''

## mf_code/external_mf.py

In [ ]:
MODULES['mf_code/external_mf.py'] = r'''
"""Apply MIMIC-adapted MF target factors to external cases."""
import os
os.environ['TF_NUM_INTRAOP_THREADS']='1';os.environ['TF_NUM_INTEROP_THREADS']='1';os.environ['TF_CPP_MIN_LOG_LEVEL']='3'
from pathlib import Path
import json,sys,time,hashlib,traceback
import numpy as np
import tensorflow as tf
H=Path(__file__).resolve().parents[1]
def main():
 checkpoint,inputpack,out,casespath,adapted=map(Path,sys.argv[1:6]);out.mkdir(parents=True,exist_ok=True);start=time.time()
 cfg=json.loads((H/'mf_adaptation.json').read_text())
 z=np.load(inputpack);a=np.load(checkpoint.with_suffix('.npz'));rng=np.random.default_rng();rng.bit_generator.state=json.loads(checkpoint.with_suffix('.json').read_text())['rng']
 u,v,ub,vb,bias=[tf.constant(a[k]) for k in ['u','v','ub','vb','bias']];nu=u.shape[0]
 ox=z['observed_x'].astype(int);oy=z['observed_y'].ravel().astype('float32');qx=z['query_x'].astype(int)
 n=int(max(ox[:,0].max(),qx[:,0].max()))+1-nu
 new_u=tf.Variable(rng.normal(0,.1,(n,64)).astype('float32'));new_b=tf.Variable(np.zeros(n,'float32'));ro=tf.keras.optimizers.Adam(.01)
 @tf.function
 def row_step():
  with tf.GradientTape() as tape:
   logits=tf.reduce_sum(tf.gather(new_u,ox[:,0]-nu)*tf.gather(v,ox[:,1]),axis=1)+tf.gather(new_b,ox[:,0]-nu)+tf.gather(vb,ox[:,1])+bias
   loss=tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=oy,logits=logits))+.001*tf.reduce_mean(new_u*new_u)
  ro.apply_gradients(zip(tape.gradient(loss,[new_u,new_b]),[new_u,new_b]))
 for _ in range(300):row_step()
 fixed_u=new_u.numpy().copy();fixed_b=new_b.numpy().copy();np.savez_compressed(out/'fixed_rows.npz',u=fixed_u,ub=fixed_b)
 fu=tf.constant(fixed_u);fb=tf.constant(fixed_b)
 cases=json.loads(casespath.read_text())
 for r in range(10):
  for case in cases:
   value=np.load(adapted/f"r{r}_{case['id']}.npz");nv=tf.constant(value['new_drug_latent']);nb=tf.constant(value['new_drug_bias'])
   prob=tf.sigmoid(tf.linalg.matvec(tf.gather(fu,qx[:,0]-nu),nv)+tf.gather(fb,qx[:,0]-nu)+nb+bias).numpy();assert np.isfinite(prob).all()
   np.savez_compressed(out/f"r{r}_{case['id']}.npz",probability=prob)
 (out/'completion.json').write_text(json.dumps(dict(passed=True,cases=len(cases),repeats=10,query_labels_read=False,target_drug_updates=0,seconds=time.time()-start,input_sha256=hashlib.sha256(inputpack.read_bytes()).hexdigest(),source_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()),indent=2))
if __name__=='__main__':
 try:main()
 except Exception:
  out=Path(sys.argv[3]);out.mkdir(parents=True,exist_ok=True);(out/'failure.json').write_text(json.dumps(dict(traceback=traceback.format_exc())));raise
'''

## classical/classifiers.py

In [ ]:
MODULES['classical/classifiers.py'] = r'''
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']:os.environ[v]='1'
os.environ['CUDA_VISIBLE_DEVICES']='-1'
from pathlib import Path
import json,time,hashlib,itertools,traceback,warnings,platform
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss
from sklearn.exceptions import ConvergenceWarning
import sklearn,xgboost as xgb,joblib
from joblib import Parallel,delayed,parallel_config
BASE=Path.cwd();records=json.loads((BASE/'records.json').read_text())
WORKERS=int(os.environ.get('CDLD_WORKERS','64'));MODE=os.environ.get('CDLD_MODE','dev')
                                     

def atomic_json(path,data):
    tmp=path.with_suffix('.tmp');tmp.write_text(json.dumps(data,indent=2));tmp.replace(path)
def score(y,p):
    v=np.clip(np.asarray(p,dtype=np.float64),1e-7,1-1e-7)
    a=float(np.mean(-(y*np.log(v)+(1-y)*np.log1p(-v))))
    assert abs(a-log_loss(y,v,labels=[0,1]))<1e-12
    return a

def run_one(rec,seed,rep,budget,chosen=None):
    start=time.monotonic();phase=rec['phase'];dest=BASE/'results'/phase/rec['name']/f's{seed}_r{rep}_b{budget}';dest.mkdir(parents=True,exist_ok=True)
    done=dest/'completion.json'
    if done.exists():
        old=json.loads(done.read_text());assert old['passed'];assert hashlib.sha256((dest/'predictions.npz').read_bytes()).hexdigest()==old['predictions_sha256'];return old
    try:
        z=np.load(BASE/rec['pack']);X=z['X'][z[f'idx_{rep}_{budget}']];y=z[f'y_{rep}_{budget}'];Q=z['X'][z['query_indices']];E=z['external_X'] if phase=='eval' else None
        assert len(y)==budget and len(X)==budget and set(np.unique(y))<={0,1}
        configs=[chosen[m][str(budget)] for m in ['LR','RF','XGB']]
        predictions={};metrics=[];fit_seconds=[];fallback=len(np.unique(y))==1;iteration_records=[]
        for ci,cfg in enumerate(configs):
            tick=time.monotonic();method=cfg['method'];steps=[cfg['round']] if method=='XGB' else [None]
            model=None
            if not fallback:
                if method=='LR':
                    model=LogisticRegression(C=cfg['C'],solver='lbfgs',max_iter=2000,tol=1e-4,random_state=seed)
                    with warnings.catch_warnings(record=True) as caught:
                        warnings.simplefilter('always',ConvergenceWarning);model.fit(X,y)
                    assert not any(issubclass(w.category,ConvergenceWarning) for w in caught),'LR convergence failure; preserve and fix numerically before resume'
                    iteration_records.append(dict(candidate=ci,iterations=model.n_iter_.tolist()))
                elif method=='RF':
                    model=RandomForestClassifier(n_estimators=300,max_depth=cfg['depth'],min_samples_leaf=cfg['leaf'],max_features='sqrt',bootstrap=True,class_weight=None,n_jobs=1,random_state=seed).fit(X,y)
                else:
                    model=xgb.XGBClassifier(n_estimators=max(steps),max_depth=cfg['depth'],learning_rate=cfg['lr'],reg_lambda=cfg['lam'],reg_alpha=0,min_child_weight=1,subsample=1,colsample_bytree=1,objective='binary:logistic',tree_method='hist',n_jobs=1,random_state=seed,base_score=.5).fit(X,y)
            for step in steps:
                ident=f'{method}_{ci}'+(f'_t{step}' if step else '')
                if fallback:p=np.full(len(Q),(float(y.sum())+1)/(len(y)+2));e=np.full(len(E),p[0]) if E is not None else None
                else:
                    kwargs={'iteration_range':(0,step)} if method=='XGB' else {}
                    p=model.predict_proba(Q,**kwargs)[:,1];e=model.predict_proba(E,**kwargs)[:,1] if E is not None and len(E) else np.empty(0) if E is not None else None
                assert np.isfinite(p).all() and np.all((p>=0)&(p<=1));predictions[ident]=p
                if E is not None:predictions[ident+'_external']=e
                entry=dict(config={**cfg,**({'round':step} if step else {})},prediction_key=ident)
                if phase=='dev':entry['bce']=score(z['query_y'],p)
                metrics.append(entry)
            fit_seconds.append(time.monotonic()-tick)
        predictions['frequency']=np.full(len(Q),float(y.mean()))
        if E is not None:predictions['frequency_external']=np.full(len(E),float(y.mean()))
        temp=dest/'predictions.tmp.npz';np.savez_compressed(temp,**predictions);temp.replace(dest/'predictions.npz')
        out=dict(passed=True,phase=phase,name=rec['name'],task=rec['task'],seed=seed,repeat=rep,budget=budget,fits=len(configs),seconds=time.monotonic()-start,fit_seconds=fit_seconds,single_class=fallback,metrics=metrics,iterations=iteration_records,predictions_sha256=hashlib.sha256((dest/'predictions.npz').read_bytes()).hexdigest(),input_sha256=hashlib.sha256((BASE/rec['pack']).read_bytes()).hexdigest(),final_query_labels_used=False)
        atomic_json(done,out);return out
    except Exception:
        atomic_json(dest/'failure.json',dict(traceback=traceback.format_exc()));raise


if __name__=='__main__':
    import sys
    seed=int(sys.argv[1]);context=sys.argv[2];chosen=json.loads((BASE/'selected_settings.json').read_text())
    rec=next(r for r in records if r['phase']=='eval' and r['name']==context)
    for rep in range(10):
        for b in [10,20,30,40,50]:run_one(rec,seed,rep,b,chosen)
    print('CLASSICAL_COMPLETE',context,seed,flush=True)
'''

## worker_models.py

In [ ]:
MODULES['worker_models.py'] = r'''
import os,sys,json,time,subprocess,traceback
from pathlib import Path
H=Path(__file__).parent;j=json.loads(sys.argv[1]);p=json.loads((H/'protocol.json').read_text());start=time.time()
os.environ.update(CUDA_VISIBLE_DEVICES='-1',TF_ENABLE_ONEDNN_OPTS='0',OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',TF_NUM_INTRAOP_THREADS='1',TF_NUM_INTEROP_THREADS='1',TF_CPP_MIN_LOG_LEVEL='3')
def call(args):subprocess.run([sys.executable]+[str(a) for a in args],check=True)
try:
 if j['model']=='MF':
  folder=H/'mf_runs'/j['context']/f"seed{j['seed']}";folder.mkdir(parents=True,exist_ok=True);inp=H/'inputs'/j['context'];checkpoint=folder/'model/checkpoint_200'
  if j['kind']=='train':call([H/'mf_code/train_mf.py',inp/'split.npz',folder/'model',j['seed'],200])
  else:
   call([H/'mf_code/adapt_mf.py',checkpoint,inp/'mimic_adapt_inputs.npz',folder/'mimic',H/'mf_cases.json'])
   call([H/'mf_code/external_mf.py',checkpoint,inp/'eicu_observed_inputs.npz',folder/'eicu',H/'mf_cases.json',folder/'mimic'])
 else:
  s=next(s for s in p['settings'] if s['id']==j['model']);os.environ.update(FINDER_L2=str(s['finder_l2']),FINDER_DROPOUT=str(s['finder_dropout_multiplier']))
  folder=H/'runs'/s['id']/j['context'];folder.mkdir(parents=True,exist_ok=True)
  for n in ['entities.json','split.npz']:
   if not (folder/n).exists():
    try:(folder/n).symlink_to(H/'inputs'/j['context']/n)
    except FileExistsError:pass
  cfg=dict(p['training'],seed=j['seed'],finder_lr=s['finder_lr'],finder_l2=s['finder_l2'],finder_dropout_multiplier=s['finder_dropout_multiplier'],finder_cycles=[s['cycle']],predictor_epochs=[s['epoch']],predictor_l2=s['predictor_l2'])
  out=folder/f"seed{j['seed']}"
  if j['kind']=='train':
   out.mkdir(exist_ok=True);(out/'requested_config.json').write_text(json.dumps(cfg));call([H/'code/train_cdld.py',folder/'split.npz',out,out/'requested_config.json'])
  else:
   base=H/'predictions'/s['id']/j['context']/f"seed{j['seed']}";args=[folder,j['seed'],s['cycle'],s['epoch']]
   call([H/'code/adapt_cdld.py']+args+[H/'inputs'/j['context']/'mimic_adapt_inputs.npz',base/'mimic',H/'cdld_adaptation.json'])
   call([H/'code/external_cdld.py']+args+[H/'inputs'/j['context']/'eicu_observed_inputs.npz',base/'eicu',H/'cdld_adaptation.json',base/'mimic'])
 (H/'done'/f"{j['id']}.json").write_text(json.dumps(dict(job=j,seconds=time.time()-start)))
except Exception:
 (H/'failures'/f"{j['id']}.json").write_text(json.dumps(dict(job=j,error=traceback.format_exc())));raise
'''

## worker_classifiers.py

In [ ]:
MODULES['worker_classifiers.py'] = r'''
from pathlib import Path
import json,sys,subprocess,time,traceback
H=Path(__file__).parent;j=json.loads(sys.argv[1]);start=time.time()
try:
 subprocess.run([sys.executable,str(H/'classical/classifiers.py'),str(j['seed']),j['context']],cwd=H/'classical',check=True)
 (H/'done'/f"{j['id']}.json").write_text(json.dumps(dict(job=j,seconds=time.time()-start)))
except Exception:
 (H/'failures'/f"{j['id']}.json").write_text(json.dumps(dict(job=j,error=traceback.format_exc())));raise
'''

## run_models.py

In [ ]:
MODULES['run_models.py'] = r'''
from pathlib import Path
import os,sys,json,subprocess,time,fcntl
H=Path(__file__).parent;lock=open(H/'runner.lock','w');fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
p=json.loads((H/'protocol.json').read_text());start=time.time();cap=max(1,min(int(os.environ.get('CDLD_WORKERS','1')),len(os.sched_getaffinity(0))))
for n in ['logs','done','failures','results']:(H/n).mkdir(exist_ok=True)
jobs=[dict(id=f'{kind}_{model}_{c}_{s}',kind=kind,model=model,context=c,seed=s) for c in p['contexts'] for s in p['seeds'] for model in ['CDLD','MF','classical'] for kind in (['eval'] if model=='classical' else ['train','eval'])]
active={}
while True:
 for jid,(proc,log) in list(active.items()):
  if proc.poll() is not None:
   log.close();del active[jid]
   if proc.returncode:raise RuntimeError('Worker failed: '+jid)
 done={x.stem for x in (H/'done').glob('*.json')}
 assert not list((H/'failures').glob('*.json')),'Preserved failures require diagnosis'
 for j in sorted(jobs,key=lambda x:x['kind']=='train'):
  if len(active)>=cap:break
  if j['id'] in done or j['id'] in active:continue
  if j['kind']=='eval' and j['model']!='classical' and j['id'].replace('eval_','train_',1) not in done:continue
  log=open(H/'logs'/f"{j['id']}.log",'a');proc=subprocess.Popen([sys.executable,str(H/('worker_classifiers.py' if j['model']=='classical' else 'worker_models.py')),json.dumps(j)],stdout=log,stderr=subprocess.STDOUT);active[j['id']]=(proc,log)
 state=dict(stage='running',elapsed_seconds=time.time()-start,total=len(jobs),complete=sum(j['id'] in done for j in jobs),active=len(active),training_complete=sum(j['kind']=='train' and j['id'] in done for j in jobs),training_total=168,evaluation_complete=sum(j['kind']=='eval' and j['id'] in done for j in jobs),evaluation_total=252)
 (H/'status.json').write_text(json.dumps(state,indent=2))
 print(f"Completed {state['complete']}/{state['total']}; active {state['active']}", flush=True)
 if not active:break
 time.sleep(15)
assert state['complete']==420
subprocess.run([sys.executable,str(H/'score.py')],check=True)
(H/'status.json').write_text(json.dumps(dict(stage='complete',complete=420,total=420,elapsed_seconds=time.time()-start)))
'''

## score.py

In [ ]:
MODULES['score.py'] = r'''
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.metrics import roc_auc_score,accuracy_score,recall_score,f1_score
H=Path(__file__).parent;P=json.loads((H/'protocol.json').read_text());OUT=H/'results';OUT.mkdir(exist_ok=True)
def metric(y,p):
 y=np.asarray(y).ravel();p=np.asarray(p,dtype=float).ravel();assert len(y)==len(p) and np.isfinite(p).all() and ((p>=0)&(p<=1)).all()
 truth=y==0;pred=p<.5;tp=int(np.sum(truth&pred));fn=int(np.sum(truth&~pred));fp=int(np.sum(~truth&pred));tn=int(np.sum(~truth&~pred));den=2*tp+fp+fn
 a=(tp+tn)/len(y);sens=tp/(tp+fn) if tp+fn else None;f=2*tp/den if den else None
 assert abs(a-accuracy_score(truth,pred))<1e-12
 if sens is not None:assert abs(sens-recall_score(truth,pred,zero_division=0))<1e-12
 if f is not None:assert abs(f-f1_score(truth,pred,zero_division=0))<1e-12
 v=np.clip(p,1e-7,1-1e-7)
 return dict(AUROC=float(roc_auc_score(truth,1-p)) if len(np.unique(y))==2 else None,accuracy=a,Sensitivity_S=tn/(tn+fp) if tn+fp else None,F1_S=2*tn/(2*tn+fp+fn) if 2*tn+fp+fn else None,Specificity=tp/(tp+fn) if tp+fn else None,TP=tn,FN=fp,FP=fn,TN=tp,BCE=float(np.mean(-y*np.log(v)-(1-y)*np.log1p(-v))))

rows=[]
for name in P['contexts']:
 labels={ds:pd.read_csv(H/'labels'/ds/name/'scoring_index.csv.gz').y.to_numpy() for ds in ['mimic','eicu']}
 for seed in P['seeds']:
  for rep in range(10):
   for b in [0,10,20,30,40,50]:
    for ds,y in labels.items():
     for candidate,arm,model in [('CDLD','CDLD','CDLD'),('MF','MF','MF')]:
      path=H/'mf_runs'/name/f'seed{seed}'/ds/f'r{rep}_MF_b{b}.npz' if candidate=='MF' else H/'predictions'/candidate/name/f'seed{seed}'/ds/f'r{rep}_{arm}_b{b}.npz'
      prob=np.load(path)['probability'];rows.append(dict(dataset=ds,combination=name,seed=seed,repeat=rep,budget=b,model=model,**metric(y,prob)))
     if b==0:
      for model in ['LR','RF','XGB','frequency']:rows.append(dict(dataset=ds,combination=name,seed=seed,repeat=rep,budget=0,model=model,**metric(y,np.full(len(y),.5))))
                                                                               
for name in P['contexts']:
 labels={ds:pd.read_csv(H/'labels'/ds/name/'scoring_index.csv.gz').y.to_numpy() for ds in ['mimic','eicu']}
 for seed in P['seeds']:
  for rep in range(10):
   for budget in [10,20,30,40,50]:
    folder=H/'classical/results/eval'/name/f's{seed}_r{rep}_b{budget}'
    meta=json.loads((folder/'completion.json').read_text());z=np.load(folder/'predictions.npz')
    for ds,y in labels.items():
     suffix='_external' if ds=='eicu' else ''
     preds={v['config']['method']:z[v['prediction_key']+suffix] for v in meta['metrics']};preds['frequency']=z['frequency'+suffix]
     for model,prob in preds.items():rows.append(dict(dataset=ds,combination=name,seed=seed,repeat=rep,budget=budget,model=model,**metric(y,prob)))
f=pd.DataFrame(rows);assert len(f)==60480
assert not f.duplicated(['dataset','combination','seed','repeat','budget','model']).any()
f.to_json(OUT/'all_conditions.json',orient='records',indent=2)
metrics=['AUROC','accuracy','Sensitivity_S','F1_S','Specificity','BCE']
comb=f.groupby(['dataset','model','combination','budget'])[metrics].mean().reset_index();comb.to_json(OUT/'combination_support.json',orient='records',indent=2)
summary=comb.groupby(['dataset','model','budget'])[metrics].mean().reset_index();summary.to_json(OUT/'support_summary.json',orient='records',indent=2)

# No target support or pretrained model is used by the four-AST mean.
ast_rows=[]
for name in P['contexts']:
 for ds,pack in [('mimic','mimic_adapt_inputs.npz'),('eicu','eicu_observed_inputs.npz')]:
  folder=H/'inputs'/name
  index=json.loads((folder/(ds+'_index.json')).read_text())
  lab=pd.read_csv(H/'labels'/ds/name/'scoring_index.csv.gz')
  assert lab.episode_id.tolist()==index['query_episodes']
  with np.load(folder/pack) as z:
   ox=z['observed_x'].astype(np.int64);oy=z['observed_y'].ravel();qr=z['query_x'][:,0].astype(np.int64)
  observed={}
  for (u,j),v in zip(ox,oy):observed.setdefault(int(u),[]).append((int(j),float(v)))
  assert all(len(observed[int(u)])==4 and len({j for j,v in observed[int(u)]})==4 for u in qr)
  prob=np.asarray([sum(v for j,v in observed[int(u)])/4 for u in qr])
  scores=metric(lab.y.to_numpy(),prob)
  scores.pop('BCE') # Five reported metrics; no calibrated-probability claim for AST mean.
  ast_rows.append(dict(dataset=ds,combination=name,model='AST mean',**scores))
a=pd.DataFrame(ast_rows);a.to_json(OUT/'ast_combination.json',orient='records',indent=2)
a.groupby(['dataset','model'])[metrics[:-1]].mean().reset_index().to_json(OUT/'ast_summary.json',orient='records',indent=2)
a.groupby(['dataset','model'])[metrics[:-1]].count().reset_index().to_json(OUT/'ast_metric_counts.json',orient='records',indent=2)
comb.groupby(['dataset','model','budget'])[metrics].count().reset_index().to_json(OUT/'metric_counts.json',orient='records',indent=2)

(OUT/'completion.json').write_text(json.dumps(dict(conditions=len(f),models=6,additional_AST_mean=True,supports=[0,10,20,30,40,50])))
'''

## Run all selected conditions / 고정 조건 실행

In [ ]:
for name, source in MODULES.items():
    dest=RUN/name; dest.parent.mkdir(parents=True,exist_ok=True); dest.write_text(source)
for name in ['protocol.json','cdld_adaptation.json','mf_cases.json','mf_training.json','mf_adaptation.json']:
    shutil.copy2(PACKAGE/'config'/name,RUN/name)
shutil.copy2(PACKAGE/'config/classical.json',RUN/'classical/selected_settings.json')
env=dict(os.environ,CDLD_WORKERS=str(WORKERS))
subprocess.run([sys.executable,str(RUN/'run_models.py')],cwd=RUN,env=env,check=True)
print('Model evaluation complete. Metrics are in the output run/results directory.')